In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from preprocessing.preprocessing_utils import preprocess_actuarial

In [2]:
# ============================================
# 1. Manual Neural Network Components
# ============================================

class DenseLayer:
    """
    A fully connected layer that manages its own weights, biases, and gradients.
    """
    def __init__(self, n_input, n_output, l2_reg=0.0):
        # He Initialization (Best practice for ReLU networks)
        # Weights are initialized with mean 0 and std = sqrt(2 / n_input)
        scale = np.sqrt(2.0 / n_input)
        self.W = np.random.randn(n_input, n_output) * scale
        self.b = np.zeros((1, n_output))
        
        self.l2_reg = l2_reg
        self.inputs = None
        self.output = None
        
        # Gradient placeholders
        self.dW = None
        self.db = None

    def forward(self, inputs):
        """
        Z = X . W + b
        """
        self.inputs = inputs
        self.output = np.dot(inputs, self.W) + self.b
        return self.output

    def backward(self, dZ):
        """
        Computes gradients dW, db and propagates error dX to previous layer.
        dZ: Gradient of Loss with respect to the output of this layer (Z).
        """
        n_samples = self.inputs.shape[0]
        
        # 1. Gradient of Weights: dW = X.T . dZ + (lambda * W)
        self.dW = np.dot(self.inputs.T, dZ)
        
        # Add L2 Regularization gradient (Weight Decay)
        if self.l2_reg > 0:
            self.dW += self.l2_reg * self.W
            
        # 2. Gradient of Biases: db = sum(dZ)
        self.db = np.sum(dZ, axis=0, keepdims=True)
        
        # 3. Gradient for previous layer: dX = dZ . W.T
        d_input = np.dot(dZ, self.W.T)
        
        return d_input

class NeuralNetworkScratch:
    def __init__(self, input_dim, hidden_dims=[128, 64], learning_rate=0.01, l2_reg=0.0001):
        self.layers = []
        self.lr = learning_rate
        
        # --- Build Architecture ---
        prev_dim = input_dim
        
        # Hidden Layers
        for h_dim in hidden_dims:
            self.layers.append(DenseLayer(prev_dim, h_dim, l2_reg))
            prev_dim = h_dim
            
        # Output Layer (Linear -> Exp is handled in loss)
        # We output 1 value (Rate)
        self.output_layer = DenseLayer(prev_dim, 1, l2_reg)
        
    def forward(self, X):
        """
        Forward pass through the network.
        Hidden Layers: ReLU(Wx + b)
        Output Layer: Exp(Wx + b) -> Poisson Rate
        """
        current_out = X
        
        # Hidden Layers (ReLU Activation)
        for layer in self.layers:
            z = layer.forward(current_out)
            # ReLU: max(0, z)
            current_out = np.maximum(0, z)
            
        # Output Layer (Linear)
        z_out = self.output_layer.forward(current_out)
        
        # Output Activation (Exponential for Poisson)
        # Clip to prevent overflow (Exploding Gradients)
        z_out = np.clip(z_out, -20, 20)
        self.predictions = np.exp(z_out)
        
        return self.predictions

    def backward(self, y_true, w_exposure):
        """
        Backpropagation.
        The gradient of Poisson Loss L = w(y_hat - y*ln(y_hat)) w.r.t z (log-link) is simply:
        dL/dz = w * (y_hat - y)
        """
        n_samples = y_true.shape[0]
        
        # 1. Output Layer Gradient
        # We normalize by batch size (n_samples) to keep gradients stable
        error = self.predictions - y_true
        delta = (w_exposure * error) / n_samples
        
        # Backprop through linear output layer
        delta = self.output_layer.backward(delta)
        
        # 2. Hidden Layers Backprop (Reverse Order)
        for i in reversed(range(len(self.layers))):
            layer = self.layers[i]
            
            # Derivative of ReLU: 1 if output > 0, else 0
            # layer.output stores the linear Z value before activation
            # But we check the input to the *next* layer (which is relu(Z))
            # Actually simpler: Just use the stored output of this layer
            relu_mask = (layer.output > 0).astype(float)
            
            delta = delta * relu_mask
            delta = layer.backward(delta)

    def step(self):
        """
        Apply Stochastic Gradient Descent (SGD) update.
        W = W - lr * dW
        """
        # Update Output Layer
        self.output_layer.W -= self.lr * self.output_layer.dW
        self.output_layer.b -= self.lr * self.output_layer.db
        
        # Update Hidden Layers
        for layer in self.layers:
            layer.W -= self.lr * layer.dW
            layer.b -= self.lr * layer.db

    def fit(self, X, y, w, epochs=50, batch_size=1024, verbose=True):
        n_samples = X.shape[0]
        loss_history = []
        
        print(f"Training on {n_samples} samples...")
        
        for epoch in range(epochs):
            # Shuffle data
            indices = np.arange(n_samples)
            np.random.shuffle(indices)
            
            epoch_loss = 0.0
            
            # Mini-batch Loop
            for start_idx in range(0, n_samples, batch_size):
                batch_idx = indices[start_idx : start_idx + batch_size]
                
                X_batch = X[batch_idx]
                y_batch = y[batch_idx]
                w_batch = w[batch_idx]
                
                # 1. Forward Pass
                preds = self.forward(X_batch)
                
                # 2. Backward Pass
                self.backward(y_batch, w_batch)
                
                # 3. Weight Update
                self.step()
                
                # 4. Track Loss (Optimization Proxy)
                # Loss = sum(w * (y_hat - y)) roughly tracks convergence
                batch_loss = np.sum(w_batch * (preds - y_batch))
                epoch_loss += batch_loss
            
            avg_loss = epoch_loss / n_samples
            loss_history.append(avg_loss)
            
            if verbose and (epoch + 1) % 10 == 0:
                print(f"Epoch {epoch+1}/{epochs} | Loss Proxy: {avg_loss:.5f}")
                
        return loss_history

    def predict(self, X):
        return self.forward(X)

In [3]:
# ============================================
# 2. Metric Functions (Manual)
# ============================================

def manual_poisson_deviance(y_true, y_pred, w):
    """
    Manual Unit Poisson Deviance.
    """
    y_pred = np.clip(y_pred, 1e-10, None)
    
    term1 = np.zeros_like(y_true)
    mask = y_true > 0
    term1[mask] = y_true[mask] * np.log(y_true[mask] / y_pred[mask])
    
    term2 = -(y_true - y_pred)
    
    deviance = 2 * w * (term1 + term2)
    return np.sum(deviance) / np.sum(w)

def manual_gini(y_true, y_pred, w):
    """
    Manual Gini Coefficient calculation.
    """
    # Sort by predicted risk (descending)
    order = np.argsort(y_pred.flatten())[::-1]
    y_true = y_true.flatten()[order]
    w = w.flatten()[order]
    
    # Cumulative Sums
    cum_w = np.cumsum(w)
    cum_w /= cum_w[-1]  # Normalize to [0, 1]
    
    cum_y = np.cumsum(y_true * w)
    cum_y /= cum_y[-1]  # Normalize to [0, 1]
    
    # Area Under Curve
    auc = np.trapz(cum_y, cum_w)
    
    # Gini = 2*AUC - 1
    return (auc - 0.5) * 2

In [4]:
# ============================================
# 3. Execution Pipeline
# ============================================

# --- A. Load Data (Reusing your preprocess logic) ---
train = pd.read_csv("../data/claims_train.csv")
test  = pd.read_csv("../data/claims_test.csv")

# Preprocess exactly like Reference M2
# IMPORTANT: Use 'scaler=None' for train, pass it to test
X_tr_df, y_tr, w_tr, scaler = preprocess_actuarial(train, scaler=None)
X_te_df, y_te, w_te, _ = preprocess_actuarial(test, scaler=scaler, ref_columns=X_tr_df.columns)

# Convert to simple NumPy (float32)
X_train = X_tr_df.values.astype(np.float32)
y_train = y_tr.values.astype(np.float32).reshape(-1, 1)
w_train = w_tr.values.astype(np.float32).reshape(-1, 1)

X_test = X_te_df.values.astype(np.float32)
y_test = y_te.values.astype(np.float32).reshape(-1, 1)
w_test = w_te.values.astype(np.float32).reshape(-1, 1)

# --- B. Configure & Train ---
# Use parameters from Reference M2 Winner:
# Architecture: [128, 64]
# LR: 0.01
# Reg: 0.0001

print("\n=== Training M2 From Scratch ===")
print(f"Architecture: [128, 64]")
print(f"Training Samples: {X_train.shape[0]}")

nn = NeuralNetworkScratch(
    input_dim=X_train.shape[1],
    hidden_dims=[128, 64],
    learning_rate=0.01,
    l2_reg=0.0001
)

start_time = time.time()
history = nn.fit(
    X_train, y_train, w_train,
    epochs=150,       # Match Reference
    batch_size=1024,  # Match Reference
    verbose=True
)
print(f"Training finished in {time.time() - start_time:.2f} seconds")

# --- C. Evaluate ---
print("\n=== Final Evaluation (Scratch M2) ===")

preds_test = nn.predict(X_test)

# 1. Deviance
test_dev = manual_poisson_deviance(y_test, preds_test, w_test)

# 2. D^2
global_mean = np.average(y_test, weights=w_test.flatten())
null_dev = manual_poisson_deviance(y_test, np.full_like(y_test, global_mean), w_test)
d2 = 1 - (test_dev / null_dev)

# 3. Gini
gini = manual_gini(y_test, preds_test, w_test)

print(f"Mean Poisson Deviance: {test_dev:.5f}")
print(f"D^2 Score: {d2:.2%}")
print(f"Gini Coefficient: {gini:.4f}")

# 4. Plot Loss
plt.figure(figsize=(8, 4))
plt.plot(history)
plt.title("Training Loss (Optimization Proxy)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True, alpha=0.3)
plt.show()


=== Training M2 From Scratch ===
Architecture: [128, 64]
Training Samples: 541416
Training on 541416 samples...
Epoch 10/150 | Loss Proxy: 0.00207
Epoch 20/150 | Loss Proxy: 0.00088
Epoch 30/150 | Loss Proxy: 0.00063
Epoch 40/150 | Loss Proxy: 0.00050
Epoch 50/150 | Loss Proxy: 0.00045
Epoch 60/150 | Loss Proxy: 0.00051
Epoch 70/150 | Loss Proxy: 0.00023
Epoch 80/150 | Loss Proxy: 0.00036
Epoch 90/150 | Loss Proxy: 0.00031
Epoch 100/150 | Loss Proxy: 0.00035
Epoch 110/150 | Loss Proxy: 0.00008
Epoch 120/150 | Loss Proxy: 0.00025
Epoch 130/150 | Loss Proxy: 0.00019
Epoch 140/150 | Loss Proxy: 0.00011
Epoch 150/150 | Loss Proxy: 0.00008
Training finished in 330.39 seconds

=== Final Evaluation (Scratch M2) ===


TypeError: Axis must be specified when shapes of a and weights differ.